# 13 — Complementos para o texto do TCC

Números e figuras pedidos pelo texto que não saíam de nenhum notebook. **Não altera nenhuma escolha**: só lê artefatos gravados pelos notebooks 01–05. Usa a série completa e as previsões do teste com finalidade *exploratória* (acesso registrado), depois da seleção congelada.  
**Grava:** `fig_funil_coorte.png`, `fig_receita_cortes.png`, `funil_coorte.csv`, `tabela_w_perfis.csv`, `mase_validacao_vs_teste.csv`, `ablacao_exogenas.csv`, `diagnostico_arimax_extremos.csv`, `diagnostico_arimax_piores_vendedores.csv`.

In [ ]:
# === SETUP (idêntico em todos os notebooks; só muda NOTEBOOK) ===
NOTEBOOK = "13_complementos_texto"
RAIZ = "/content/drive/MyDrive/tcc_fluxo"      # Colab. Localmente: caminho da pasta do projeto (ou deixe None)

import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q -r {RAIZ}/requirements.txt
except ImportError:                              # fora do Colab
    RAIZ = os.environ.get("TCC_RAIZ") or None
    if RAIZ is None:
        from pathlib import Path
        RAIZ = str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").is_file()))
sys.path.insert(0, f"{RAIZ}/src")
%load_ext autoreload
%autoreload 2

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tcc_fluxo import iniciar
C, A = iniciar(RAIZ, notebook=NOTEBOOK)

In [ ]:
from tcc_fluxo import complementos as CP, painel as P, graficos
from tcc_fluxo.cv import origens_cv

sel = A.carregar_json("selecao_modelo")
CHOSEN = sel["modelo_escolhido"]
weekly_observed = A.carregar_df("weekly_observed")
perfis = A.carregar_df("perfis_treino")
hist = A.carregar_historico_treino()
serie = A.carregar_serie_completa(finalidade="exploratoria")
comparison = A.carregar_df("previsoes_teste")
cv_det = A.carregar_df("cv_detalhe")
resumo_val = A.carregar_df("resumo_validacao")
TEST_END = pd.Timestamp(A.carregar_json("corte")["TEST_END"])
ORIGENS, _ = origens_cv(C)
print("Modelo escolhido:", CHOSEN)

## 1. Funil da coorte (seção 4.2)

In [ ]:
elig, cohort, _, _ = P.coorte(weekly_observed, C.PANEL_START, C.TRAIN_END, C.MIN_CALENDAR_WEEKS,
                              C.MIN_SALES_WEEKS, C.USE_MEI_PROXY, C.MEI_MONTHLY_PROXY_LIMIT)
etapas, extra = CP.funil(elig, weekly_observed, C.TRAIN_END, C.MIN_CALENDAR_WEEKS, C.MIN_SALES_WEEKS)
assert etapas["vendedores"].iloc[-1] == len(cohort) == perfis["seller_id"].nunique()
A.tabela(etapas.assign(**{k: v for k, v in extra.items()}), "funil_coorte.csv")
display(etapas); print(pd.Series(extra))

fig, ax = plt.subplots(figsize=(9, 3.4))
rot = list(etapas["etapa"]) + ["Com venda só em 2018 (fora da coorte)"]
val = list(etapas["vendedores"]) + [extra["vendedores_so_em_2018 (fora por construção)"]]
cores = [graficos.C_OTHER] * (len(rot) - 2) + [graficos.C_CHOSEN, "#dedcd5"]
y = np.arange(len(rot))[::-1]
ax.barh(y, val, color=cores, height=0.6)
for yi, v, (_, r) in zip(y, val, pd.concat([etapas, pd.DataFrame([{"excluidos_nesta_etapa": 0}])]).iterrows()):
    txt = f"{v:,}".replace(",", ".")
    if r["excluidos_nesta_etapa"]:
        txt += f"  (−{int(r['excluidos_nesta_etapa']):,})".replace(",", ".")
    ax.text(v + max(val) * 0.01, yi, txt, va="center", fontsize=9)
ax.set_yticks(y, rot, fontsize=9); ax.set_xlim(0, max(val) * 1.25); ax.set_xticks([])
for sp in ["top", "right", "bottom"]: ax.spines[sp].set_visible(False)
ax.set_title("Seleção da coorte de vendedores", loc="left", fontsize=10)
fig.tight_layout(); A.figura(fig, "fig_funil_coorte.png"); plt.show()

## 2. Receita semanal com os cortes de treino, validação e teste (seção 4.5)

In [ ]:
tot = serie.groupby("week_start")["revenue"].sum()
# A receita da semana w é o alvo da linha da semana w-1: os cortes abaixo são em semana-ALVO.
primeiro_alvo = A.carregar_treino()["target_week"].min()
fig, ax = plt.subplots(figsize=(12, 4.4))
ax.axvspan(tot.index.min() - pd.Timedelta(days=3), primeiro_alvo - pd.Timedelta(days=4), color="#e5e7eb", lw=0,
           label="Aquecimento (só atributos: 8 semanas de defasagem)")
ax.axvspan(primeiro_alvo - pd.Timedelta(days=3), C.TRAIN_TARGET_END + pd.Timedelta(days=3), color="#dbeafe", lw=0,
           label="Treino (alvos 06/03 a 25/12/2017)")
for i, o in enumerate(ORIGENS):
    ax.axvspan(o + pd.Timedelta(days=4), o + pd.Timedelta(weeks=C.CV_VAL_WEEKS, days=3), ymin=0, ymax=0.06,
               color="#2563eb", lw=0, label="Janelas de validação (6 origens × 4 semanas)" if i == 0 else None)
ax.axvspan(C.TRAIN_TARGET_END + pd.Timedelta(days=4), TEST_END + pd.Timedelta(days=3), color="#fde8dc", lw=0,
           label=f"Teste (alvos 01/01 a {TEST_END:%d/%m/%Y})")
ax.plot(tot.index, tot.values, color=graficos.INK, lw=1.8)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v / 1000:,.0f} mil"))
ax.set_ylabel("R$ por semana"); ax.set_ylim(0, tot.max() * 1.12); graficos.clean(ax)
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax.set_title("Receita semanal da coorte (870 vendedores) e divisão temporal", loc="left", fontsize=10)
fig.tight_layout(); A.figura(fig, "fig_receita_cortes.png"); plt.show()

## 3. Tabela W — perfis de vendedor (seção 5.1)

In [ ]:
tab_w = CP.tabela_perfis(serie, perfis, C.TRAIN_TARGET_END)
A.tabela(tab_w, "tabela_w_perfis.csv"); display(tab_w.round(1))

## 4. Por que o MASE de validação é maior que o de teste (seções 4.7 e 5.2)\nMASE = MAE ÷ escala, com escala = média de |Δ semanal| da receita do próprio vendedor até o corte. Duas coisas mudam entre validação e teste: (i) o **numerador** — as semanas de validação (jul–dez/2017, com a Black Friday) são mais difíceis que as de 2018 (veja `MAE_Naive_R$`); (ii) o **denominador** — nas origens iniciais há vendedores com pouco histórico e escala pequena, que inflam a média de MASE por vendedor (veja `media_1_sobre_escala`). Como as duas coisas afetam o Naive e o modelo igualmente, a `razao_escolhido_sobre_Naive` fica estável: a queda não indica sobreajuste nem vazamento.

In [ ]:
vt = CP.validacao_vs_teste(cv_det, resumo_val, comparison, hist, ORIGENS, C.TRAIN_TARGET_END, ["Naive", CHOSEN])
A.tabela(vt, "mase_validacao_vs_teste.csv"); display(vt.round(3))

## 5. Ablação: modelo escolhido sem × com exógenas (seção 5.6)

In [ ]:
SEM = CHOSEN.removesuffix("_Exog")
assert f"pred_{SEM}" in comparison.columns, f"{SEM} sem previsões no teste"
abl = CP.ablacao(comparison, hist, C.TRAIN_TARGET_END, SEM, CHOSEN, C.N_BOOT, C.RANDOM_SEED)
A.tabela(abl, "ablacao_exogenas.csv"); display(abl.round(4))
print("p > 0,05 em todas as linhas = sem diferença significativa entre os modelos com e sem exógenas.")

## 6. ARIMAX/SARIMAX por vendedor: previsões extremas (seção 5.7)\nPor que as versões com exógenas ajustadas por vendedor e somadas têm erro agregado tão alto: (i) vendedores cuja previsão máxima no teste supera 10× (ou 5×) a maior receita semanal do próprio vendedor no treino, e quanto do erro vem deles; (ii) efeito do corte das previsões negativas em zero.

In [ ]:
pred_sec = A.carregar_df("secundaria_pred_vendedor_estat")     # previsões brutas da 08a
arq = A.saidas / "secundaria_metricas_agregado.csv"
mase_ref = None
if arq.exists():
    t = pd.read_csv(arq)
    t = t.loc[t["modelo"].str.endswith("(soma por vendedor)")]
    mase_ref = dict(zip(t["modelo"].str.replace(" (soma por vendedor)", "", regex=False), t["MASE"]))
diag_ext, casos_ext = CP.diagnostico_estat_extremos(pred_sec, comparison, hist, perfis, mase_ref)
A.tabela(diag_ext, "diagnostico_arimax_extremos.csv"); A.tabela(casos_ext, "diagnostico_arimax_piores_vendedores.csv")
display(diag_ext.round(3)); display(casos_ext.round(1))

In [ ]:
A.finalizar()